# 03 - Application du Modèle de Motion Flow
Ce notebook charge un modèle de Motion Flow entraîné et l'applique à une image statique pour générer une vidéo ou un GIF animé.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
from scipy.ndimage import map_coordinates

# Ajouter le répertoire parent au path pour pouvoir importer depuis src
sys.path.append(os.path.abspath('..'))
from motion_flow.model import MotionFlowUNet

In [ ]:
import glob

# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilisation du device: {device}")

TARGET_SIZE = 512

# Recherche automatique du dernier checkpoint
ckpt_dir = "../checkpoints"
CHECKPOINT_PATH = ""

if os.path.exists(ckpt_dir):
    runs = sorted(glob.glob(os.path.join(ckpt_dir, "run_*")), reverse=True)
    if runs:
        latest_run = runs[0]
        # On tente de charger model_best en priorité
        model_path = os.path.join(latest_run, "model_best.pth")
        if not os.path.exists(model_path):
            model_path = os.path.join(latest_run, "model_latest.pth")
            
        if os.path.exists(model_path):
            CHECKPOINT_PATH = model_path
            print(f"✓ Meilleur/Dernier modèle trouvé automatiquement : {CHECKPOINT_PATH}")
        else:
            print(f"⚠️ Aucun modèle .pth trouvé dans le run {latest_run}")
    else:
        print("⚠️ Aucun run trouvé dans checkpoints/")
else:
    print("⚠️ Dossier checkpoints/ introuvable")

if not CHECKPOINT_PATH:
    print("⚠️ Attention: Vous devez avoir un modèle entraîné pour que l'inférence soit pertinente !")


In [ ]:
# 1. Chargement du modèle
if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
    model = MotionFlowUNet().to(device)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    model.eval()
    print("Modèle chargé et mis en mode évaluation.")

## 2. Chargement et Prétraitement de l'Image

In [ ]:
import random
from data.dataset import LandscapeMotionDataset

DATA_DIR = '../data/youtube_landscape'

# Fonction pour redimensionner au carré comme lors de l'entraînement
def preprocess_for_inference(img_array, target_size=512):
    h, w = img_array.shape[:2]
    # Resize
    if h < w:
        new_h = target_size
        new_w = int(w * (target_size / h))
    else:
        new_w = target_size
        new_h = int(h * (target_size / w))
    
    resized = cv2.resize(img_array, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # Center Crop
    start_y = (new_h - target_size) // 2
    start_x = (new_w - target_size) // 2
    cropped = resized[start_y:start_y+target_size, start_x:start_x+target_size]
    
    # Normalisation entre 0 et 1 (comme transforms.ToTensor())
    tensor = torch.from_numpy(cropped).permute(2, 0, 1).float() / 255.0
    # Ajouter la dimension de batch: (1, C, H, W)
    return tensor.unsqueeze(0), cropped

try:
    dataset = LandscapeMotionDataset(DATA_DIR)
    if len(dataset) > 0:
        idx = random.randint(0, len(dataset) - 1)
        print(f"Tirage aléatoire de l'image index: {idx}")
        img_A, _ = dataset[idx]
        
        # Le dataset retourne déjà un Tensor (C, H, W) entre 0.0 et 1.0 (transforms.ToTensor())
        # on repasse en numpy pour le plot et on rajoute juste la dimension Batch
        input_tensor = img_A.unsqueeze(0).to(device)
        cropped_img = img_A.permute(1, 2, 0).numpy() # (H, W, C) pour plt
        
        plt.figure(figsize=(6, 6))
        plt.imshow(cropped_img)
        plt.title("Image originale du Dataset")
        plt.axis('off')
        plt.show()
    else:
        print("Dataset vide.")
except Exception as e:
    print(f"Erreur de chargement: {e}")

## 3. Prédiction du Mouvement (Inference)

In [ ]:
if 'model' in locals() and 'input_tensor' in locals():
    with torch.no_grad():
        # Notre modèle prédit un tenseur de mouvement (1, 2, H, W)
        pred_flow = model(input_tensor)
        
    flow = pred_flow.squeeze(0).cpu().numpy() # (2, H, W)
    flow_x = flow[0]
    flow_y = flow[1]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    im0 = axes[0].imshow(flow_x, cmap='coolwarm')
    axes[0].set_title("Prédiction Motion Flow (U - axe X)")
    fig.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].imshow(flow_y, cmap='coolwarm')
    axes[1].set_title("Prédiction Motion Flow (V - axe Y)")
    fig.colorbar(im1, ax=axes[1])
    plt.show()

## 4. Animation (Warping)
On applique le flux à notre image pour synthétiser un mouvement, puis on sauve ce résultat en GIF.

In [ ]:
if 'cropped_img' in locals() and 'flow_x' in locals():
    h, w, c = cropped_img.shape
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    
    frames = []
    num_frames = 30
    
    # On va simuler un mouvement fluide en faisant varier l'amplitude du flow prédit 
    for i in range(num_frames):
        # Ex: Mouvement d'aller-retour avec un sinus
        amplitude = np.sin((i / num_frames) * 2 * np.pi) 
        
        # Mouvement accumulé
        map_y = y - (flow_y * amplitude * 10) # * 10 pour amplifier l'effet si le modèle prédit de petites valeurs
        map_x = x - (flow_x * amplitude * 10)
        
        frame = np.zeros_like(cropped_img)
        
        for channel in range(c):
            frame[..., channel] = map_coordinates(
                cropped_img[..., channel], 
                [map_y, map_x], 
                order=1, 
                mode='reflect'
            )
            
        frames.append(Image.fromarray(frame))
        
    gif_path = '../data/gif.gif'
    frames[0].save(gif_path,
                   save_all=True,
                   append_images=frames[1:],
                   duration=100,
                   loop=0)
                   
    print(f"GIF généré: {gif_path}")
    
    # Affichage
    from IPython.display import Image as IPyImage, display
    display(IPyImage(url=gif_path))